# Train Mario (Phase 2) — watch it learn inside Jupyter

This trains a **PPO** agent and shows everything in the page:
1. training runs and the progress tables scroll live
2. the agent's "brain" (the neural-network weights) is saved to a `.zip`
3. we replay the trained agent as an embedded video

**The "brain"** = the weights of a CNN that maps the 84x84x4 image to a button
press. Saving it = `model.save(path)` -> a `.zip`. Reload later with
`PPO.load(path)` (no need to retrain).

In [ ]:
import warnings; warnings.filterwarnings("ignore")
from stable_baselines3 import PPO
from game_env import make_venv
from games import get_game

# Small on purpose so a cell finishes in a few minutes. Bump to 500_000+ for a
# Mario that actually gets somewhere (that's hours on CPU).
TIMESTEPS = 10_000

venv = make_venv(get_game("mario"), 1)
model = PPO(
    "CnnPolicy", venv, verbose=1,
    n_steps=512, batch_size=64, n_epochs=10,
    learning_rate=1e-4, gamma=0.9, ent_coef=0.01,
    device="cpu",
)

In [ ]:
# Watch the tables scroll: ep_rew_mean should trend up, explained_variance
# should climb above 0 (the value network is learning).
model.learn(total_timesteps=TIMESTEPS)

In [ ]:
# Save the brain
import os
os.makedirs("../data/models", exist_ok=True)
path = "../data/models/mario_ppo_notebook"
model.save(path)
print("saved brain:", path + ".zip")
print("reload later with: PPO.load(\"" + path + ".zip\")")

## Watch the trained agent play

In [ ]:
import imageio
from IPython.display import Video
from stable_baselines3.common.vec_env import VecTransposeImage

# SB3 trains on channels-first internally, so match that layout for prediction.
rec = VecTransposeImage(make_venv(get_game("mario"), 1))
obs = rec.reset()
frames = [rec.get_attr("last_rgb")[0].copy()]
done, steps, max_x = False, 0, 0
while not done and steps < 3000:
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, dones, infos = rec.step(action)
    frames.append(rec.get_attr("last_rgb")[0].copy())
    max_x = max(max_x, infos[0].get("x_pos", 0))
    done = bool(dones[0]); steps += 1
rec.close()

OUT = "../data/mario_ppo.mp4"
imageio.mimsave(OUT, frames, fps=30)
print(f"episode: {steps} steps, furthest x_pos: {max_x}")
Video(OUT, embed=True, width=384)